In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

import sklearn.decomposition as dp
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split

In [ ]:
import sklearn
import cloudpickle
sklearn.__version__

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)


In [ ]:
granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))
#X = X[indx_tot]


In [ ]:
X_train,X_test = train_test_split(X,test_size=0.2,random_state=42)

In [ ]:
recon_losses_train = np.zeros(20)
recon_losses_test = np.zeros(20)

model_nmf_list = []

for i in range(20):
    print(i)
    model_nmf = dp.NMF(int(i+1),max_iter=1000)
    S_train = model_nmf.fit_transform(X_train)
    S_test = model_nmf.transform(X_test)
    X_r_train = np.dot(S_train,model_nmf.components_)
    X_r_test = np.dot(S_test,model_nmf.components_)
    recon_losses_train[i] = np.mean((X_train-X_r_train)**2)
    recon_losses_test[i] = np.mean((X_test-X_r_test)**2)
    model_nmf_list.append(model_nmf)

In [ ]:
cloudpickle.dump({'models':model_nmf_list},open('Models.p','wb'))

# In this plot we are looking for an "ELBOW"

In [ ]:
plt.plot(recon_losses_test)

In [ ]:
np.mean((X_test-np.mean(X_test,axis=0))**2)

In [ ]:
np.savetxt('NFM_recon_train.txt',recon_losses_train,fmt='%0.8f',delimiter='\n')
np.savetxt('NFM_recon_test.txt',recon_losses_test,fmt='%0.8f',delimiter='\n')

## Now compute BIC

In [ ]:
recon_losses_train = np.genfromtxt('NFM_recon_train.txt')
recon_losses_test = np.genfromtxt('NFM_recon_test.txt')

In [ ]:
def BIC(N,p,i,loss):
    part1 = i*p*np.log(N)
    part2 = loss*N*p #
    print(part1,part2)
    bic = part1+2*part2
    return bic,part1,part2

def AIC(N,p,i,loss):
    part1 = i*p*np.log(N) # We wa
    part2 = loss*N*p
    bic = part1+part2
    return bic,part1,part2

In [ ]:
p = X.shape[1]
N = X_test.shape[0]

In [ ]:
print(N,p)

In [ ]:
bics = np.zeros(len(recon_losses_train))
p1s = np.zeros(len(recon_losses_train))
p2s = np.zeros(len(recon_losses_train))

for i in range(len(recon_losses_train)):
    bics[i],p1s[i],p2s[i] = BIC(N,p,i+1,recon_losses_test[i])
plt.plot(p1s+2*p2s)
plt.plot(bics)

In [ ]:
aics = np.zeros(len(recon_losses_train))
p1s = np.zeros(len(recon_losses_train))
p2s = np.zeros(len(recon_losses_train))

for i in range(len(recon_losses_train)):
    aics[i],p1s[i],p2s[i] = BIC(N,p,i+1,recon_losses_test[i])
plt.plot(np.arange(20)+1,p1s+p2s)

In [ ]:
plt.plot(np.arange(20)+1,aics)

In [ ]:
def AIC(N,p,i,loss):
    part1 = i*p*np.log(N) # We wa
    part2 = loss*N*p
    bic = part1+part2
    return bic,part1,part2